# Notebook 5 — Simulation Quality Assessment

Evaluates how well a driving simulator reproduces the statistical properties
of real Waymo on-road data.

This is critical for AV safety: if the simulator under-represents rare or
dangerous scenarios, safety guarantees derived from simulation do not transfer
to the real world.

**Setup:** This notebook compares two splits of the Waymo validation set as
a proxy for real vs. simulated data. In a real workflow, replace
`sim_trajectories` with output from your simulator.

Metrics:
- Kolmogorov-Smirnov test
- Jensen-Shannon Divergence
- Wasserstein distance
- Maximum Mean Discrepancy (MMD)
- Coverage / diversity score

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.waymo_loader import load_dataset, extract_trajectories
from src.metrics.safety_metrics import nearest_agent_distances
from src.simulation.quality_metrics import compare_distributions
from src.visualization.plots import plot_distribution_comparison, plot_sim_quality_summary

In [2]:
dataset = load_dataset('../data', max_segments=10)
trajectories = extract_trajectories(dataset['lidar_box'])
trajectories = nearest_agent_distances(trajectories)

# Split into two halves as real / sim proxy
segments = trajectories['segment_id'].unique()
mid = len(segments) // 2
real_traj = trajectories[trajectories['segment_id'].isin(segments[:mid])]
sim_traj = trajectories[trajectories['segment_id'].isin(segments[mid:])]

print(f'Real split: {len(real_traj):,} rows from {real_traj["segment_id"].nunique()} segments')
print(f'Sim split:  {len(sim_traj):,} rows from {sim_traj["segment_id"].nunique()} segments')

Real split: 62,312 rows from 5 segments
Sim split:  46,381 rows from 5 segments


## Per-Feature Distribution Comparison

In [3]:
for feat in ['speed', 'acceleration', 'nearest_agent_dist']:
    if feat in real_traj.columns:
        fig = plot_distribution_comparison(real_traj[feat], sim_traj[feat], feat)
        plt.show()

/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_28046/2363402930.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_28046/2363402930.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_28046/2363402930.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Full Simulation Quality Report

In [4]:
report = compare_distributions(real_traj, sim_traj)
print(f'Overall match score: {report.overall_score:.2%}')
print(f'MMD: {report.mmd:.6f}')
print(f'Coverage score: {report.coverage_score:.2%}')

report.summary_df().set_index('feature').style.background_gradient(
    cmap='RdYlGn', subset=['match_score']
).background_gradient(
    cmap='RdYlGn_r', subset=['jsd', 'ks_statistic']
)

Overall match score: 98.93%
MMD: 0.005590
Coverage score: 100.00%


,ks_statistic,ks_pvalue,jsd,wasserstein_dist,mean_bias,std_ratio,match_score
feature,,,,,,,
speed,0.226280,0.000000,0.027899,0.490178,-0.489580,0.720714,0.972101
acceleration,0.133851,0.000000,0.004013,0.034374,-0.031129,0.943164,0.995987
jerk,0.066151,0.000000,0.003171,0.029902,-0.000900,0.920762,0.996829
heading_rate,0.141859,0.000000,0.000586,0.074812,0.003614,0.907417,0.999414
nearest_agent_dist,0.098035,0.000000,0.018069,0.613201,0.460441,1.221037,0.981931
OVERALL,nan,nan,nan,nan,nan,nan,0.989252


In [5]:
fig = plot_sim_quality_summary(report)
plt.show()

/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_28046/4095044400.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Interpretation

| Metric | Ideal | Acceptable |
|--------|-------|------------|
| JSD | 0.0 | < 0.10 |
| Wasserstein | 0.0 | domain-dependent |
| Match Score | 1.0 | > 0.90 |
| Coverage | 1.0 | > 0.95 |
| MMD | 0.0 | < 0.01 |

When JSD or Wasserstein distance is high for a specific feature, the simulator
is not faithfully reproducing that aspect of real-world driving.  Prioritise
calibrating the simulator on features with the lowest match score.